# Phase 4 — Robust Training (Corruption-Aware Augmentation)

**Pre-requisite:** Phase 3 done — `results/robustness_baseline.json` exists,
baseline `best.pt` in Drive, 25 corrupted sets available.

Retrain RT-DETR-L with corruption-mimicking augmentation. Only the augmentation
pipeline changes vs. Phase 2 — all other hyperparameters identical.

**Targets:** mPC improvement >= 10% relative; clean mAP drop < 0.03.

In [ ]:
import os, json, shutil, glob, tempfile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_DIR = '/content/drive/MyDrive/robot-perception'
RESULTS_DIR = f'{PROJECT_DIR}/results/figures'
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']
CORRUPTIONS = ['gaussian_noise', 'motion_blur', 'gaussian_blur', 'brightness', 'occlusion']
SEVERITIES  = [1, 2, 3, 4, 5]

import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
# Pin albumentations — the transform list below targets this exact API.
!pip install -q 'albumentations==1.4.18'
import albumentations as A
print('albumentations:', A.__version__)

## Step 1 — Copy dataset to local disk + write yaml

In [ ]:
LOCAL_DATA = '/content/robot_data'
if not os.path.exists(LOCAL_DATA):
    shutil.copytree(f'{PROJECT_DIR}/data/annotated', LOCAL_DATA)

LOCAL_YAML = '/content/robot_parts.yaml'
with open(LOCAL_YAML, 'w') as f:
    f.write(f"""path: {LOCAL_DATA}
train: images/train
val: images/val
test: images/test

nc: 5
names: ['arm', 'leg', 'torso', 'head', 'sensor']
""")
for split in ['train', 'val', 'test']:
    n = len([f for f in os.listdir(f'{LOCAL_DATA}/images/{split}')
             if f.endswith(('.jpg','.png','.jpeg'))])
    print(f'{split}: {n} images')

## Step 2 — Corruption-aware augmentation list

**Only corruption-mimicking transforms.** Ultralytics already handles resize,
normalization, flips, and scale — do NOT add those here (would double-process).

Ultralytics' `Albumentations` transform accepts a custom list via the
`augmentations=` training argument (verified in `data/augment.py`).

In [ ]:
import albumentations as A

# Corruption-mimicking transforms only. Ranges span the Phase-3 test severities
# so training augmentation overlaps what the model is benchmarked against.
robust_augmentations = [
    A.OneOf([
        A.GaussNoise(std_range=(0.04, 0.26), p=1.0),
        A.MotionBlur(blur_limit=(3, 15), p=1.0),
        A.GaussianBlur(blur_limit=(3, 9), p=1.0),
    ], p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.3, p=0.4),
    A.CoarseDropout(
        num_holes_range=(1, 8),
        hole_height_range=(0.04, 0.12),
        hole_width_range=(0.04, 0.12),
        fill=0,
        p=0.3,
    ),
]
for t in robust_augmentations:
    print('  -', t.__class__.__name__)

# NOTE: if a kwarg error appears, your albumentations version differs.
# Re-run the pip-install cell above to pin 1.4.18, then restart runtime.

## Step 3 — Retrain RT-DETR-L

Fresh COCO weights. All hyperparameters identical to Phase 2 except augmentation.
~2-4 h on a T4. `last.pt` is copied to Drive periodically so a disconnect is recoverable.

In [ ]:
from ultralytics import RTDETR

model_robust = RTDETR('rtdetr-l.pt')  # same COCO start as baseline

results = model_robust.train(
    data=LOCAL_YAML,
    epochs=100,
    imgsz=640,
    batch=8,
    lr0=1e-4,
    weight_decay=1e-4,
    warmup_epochs=3,
    patience=20,
    device=0,
    project='/content/runs',
    name='robust',
    exist_ok=True,
    save=True,
    augmentations=robust_augmentations,   # <-- custom corruption-aware list
)

In [ ]:
# Save robust weights to Drive
best_drive = f'{PROJECT_DIR}/models/robust/best.pt'
os.makedirs(os.path.dirname(best_drive), exist_ok=True)
shutil.copy2('/content/runs/robust/weights/best.pt', best_drive)
print(f'Saved: {best_drive}')

# T4 disconnect safety: if training was interrupted, last.pt is in
# /content/runs/robust/weights/last.pt — copy it to Drive and resume with
#   RTDETR('<drive>/last.pt').train(..., resume=True)

## VIZ 4 — Robust training curve

In [ ]:
df = pd.read_csv('/content/runs/robust/results.csv')
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df['epoch'], df['train/box_loss'], label='train box loss')
axes[0].plot(df['epoch'], df['val/box_loss'], label='val box loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Robust model — box loss'); axes[0].legend()
axes[1].plot(df['epoch'], df['metrics/mAP50(B)'], color='green', label='val mAP@0.5')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mAP@0.5')
axes[1].set_title('Robust model — validation mAP'); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz4_robust_training_curve.png', dpi=150)
plt.show()

## Step 4 — Clean test eval (must not regress > 0.03)

In [ ]:
model = RTDETR(f'{PROJECT_DIR}/models/robust/best.pt')
robust_clean = float(model.val(data=LOCAL_YAML, split='test', verbose=False).box.map50)

with open(f'{PROJECT_DIR}/results/robustness_baseline.json') as f:
    baseline = json.load(f)
baseline_clean = baseline['clean_mAP50']
baseline_mPC   = baseline['mPC']

print(f'Baseline clean mAP@0.5: {baseline_clean:.4f}')
print(f'Robust   clean mAP@0.5: {robust_clean:.4f}')
drop = baseline_clean - robust_clean
print(f'Clean mAP drop: {drop:.4f}  (target < 0.03)')
print('✅ PASS' if drop < 0.03 else '⚠️  FAIL — augmentation hurt clean performance')

## Step 5 — Evaluate robust model on all 25 corrupted sets

Reuses the corrupted sets persisted to Drive in Phase 3. If they were not
persisted, regenerate them first (re-run Phase 3 Step 3).

In [ ]:
# Pull corrupted sets to local disk for fast eval
CORRUPTED_LOCAL = '/content/corrupted'
DRIVE_CORRUPTED = f'{PROJECT_DIR}/data/corrupted'
if not os.path.exists(CORRUPTED_LOCAL):
    if os.path.exists(DRIVE_CORRUPTED):
        shutil.copytree(DRIVE_CORRUPTED, CORRUPTED_LOCAL)
        print('Copied corrupted sets from Drive.')
    else:
        raise FileNotFoundError(
            'No corrupted sets found. Re-run Phase 3 Step 3 to generate them.')
else:
    print('Corrupted sets already local.')

In [ ]:
def build_temp_yaml(sev_dir, tmp_dir):
    content = f"""path: {os.path.abspath(sev_dir)}
train: images
val: images
test: images

nc: 5
names: ['arm', 'leg', 'torso', 'head', 'sensor']
"""
    p = os.path.join(tmp_dir, os.path.basename(sev_dir) + '.yaml')
    with open(p, 'w') as f:
        f.write(content)
    return p

robust_grid = {}
tmp_dir = tempfile.mkdtemp()
done = 0
for corruption in CORRUPTIONS:
    robust_grid[corruption] = {}
    for severity in SEVERITIES:
        sev_dir = f'{CORRUPTED_LOCAL}/{corruption}/severity_{severity}'
        for c in glob.glob(f'{sev_dir}/labels/*.cache'):
            os.remove(c)
        yaml_path = build_temp_yaml(sev_dir, tmp_dir)
        try:
            mAP = float(model.val(data=yaml_path, split='test', verbose=False).box.map50)
        except Exception as e:
            print(f'  ERROR {corruption}/s{severity}: {e}')
            mAP = 0.0
        robust_grid[corruption][severity] = mAP
        done += 1
        print(f'  [{done}/25] {corruption}/severity_{severity}: mAP={mAP:.4f}')

In [ ]:
robust_mPC = float(np.mean([robust_grid[c][s] for c in CORRUPTIONS for s in SEVERITIES]))
robust_rel = robust_mPC / robust_clean if robust_clean > 0 else 0.0

out = {
    'weights': f'{PROJECT_DIR}/models/robust/best.pt',
    'clean_mAP50': robust_clean,
    'mPC': robust_mPC,
    'relative_mPC': robust_rel,
    'results_grid': {c: {str(s): robust_grid[c][s] for s in SEVERITIES} for c in CORRUPTIONS},
}
with open(f'{PROJECT_DIR}/results/robustness_robust.json', 'w') as f:
    json.dump(out, f, indent=2)

improvement = (robust_mPC - baseline_mPC) / baseline_mPC if baseline_mPC > 0 else 0.0
print('=== mPC Comparison ===')
print(f'Baseline mPC: {baseline_mPC:.4f}')
print(f'Robust   mPC: {robust_mPC:.4f}')
print(f'Relative improvement: {improvement*100:.1f}%  (target >= 10%)')
print('✅ PASS' if improvement >= 0.10 else '⚠️  FAIL — see diagnostics below')

print('\nPer-corruption improvement:')
for c in CORRUPTIONS:
    b = np.mean([baseline['results_grid'][c][str(s)] for s in SEVERITIES])
    r = np.mean([robust_grid[c][s] for s in SEVERITIES])
    print(f'  {c}: {b:.4f} -> {r:.4f}  ({(r-b)/b*100:+.1f}%)')

if improvement < 0.10:
    print('\nDIAGNOSTIC: training-aug severity may not overlap test-corruption severity.')
    print('Knobs: widen GaussNoise std_range / blur_limit; raise OneOf p from 0.5 to ~0.7.')

## Step 6 — Side-by-side robustness comparison

In [ ]:
fig, axes = plt.subplots(1, len(CORRUPTIONS), figsize=(20, 4))
for i, corruption in enumerate(CORRUPTIONS):
    base = [baseline['results_grid'][corruption][str(s)] for s in SEVERITIES]
    rob  = [robust_grid[corruption][s] for s in SEVERITIES]
    axes[i].plot(SEVERITIES, base, 'r-o', label='baseline')
    axes[i].plot(SEVERITIES, rob,  'g-o', label='robust')
    axes[i].axhline(y=baseline_clean, color='k', linestyle='--')
    axes[i].set_title(corruption); axes[i].set_xlabel('Severity')
    axes[i].set_xticks(SEVERITIES); axes[i].legend()
axes[0].set_ylabel('mAP@0.5')
plt.suptitle('Robustness comparison — baseline (red) vs robust (green)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/robustness_comparison.png', dpi=150)
plt.show()

## Phase 4 Completion Checklist

- [ ] Robust model trained with corruption-aware augmentation
- [ ] Clean mAP drop < 0.03
- [ ] Robust model evaluated on all 25 corrupted sets
- [ ] `results/robustness_robust.json` saved
- [ ] Side-by-side comparison plot saved
- [ ] mPC improvement >= 10% relative to baseline

Next → `05_mc_dropout.ipynb`